# Figures for gravity.md

*David Munro*


In [6]:
# General imports and utility functions

from numpy import (array, asarray, pi, sin, cos, sqrt, linspace, zeros,
    digitize, maximum, minimum, arange, arctan2, concatenate, interp)


# Two argument sinc for when sin(x) has already been computed.
def sinc2(x, sinx):
    """return sinx/x, assuming sinx = sin(x)"""
    x = asarray(x, float)  # incorrect if x is complex
    sinx = asarray(sinx, float)
    zero = (x == 0.) + 0.  # 1.0 where x == 0, else 0.0
    return (zero + sinx) / (zero + x)  # sinx==x exactly if abs(x) < 1e-8


# polyconic function is for reference only - essentially this is case of
# ncone=infinity in the cone_map function below
def polyconic(phi, theta, full=False):
    """Polyconic mapping from (lon, lat) in radians to (x, y)"""
    phi, theta = asarray(phi, float), asarray(theta, float)
    sint, cost = sin(theta), cos(theta)
    fphi = 0.5 * sint * phi
    sinf, cosf = sin(fphi), cos(fphi)
    sincf = sinc2(fphi, sinf)
    r = phi * cost * sincf
    x, y = r * cosf, theta + r * sinf
    if not full:
        return x, y
    # Also compute normalized direction from (x, y) to center of lat circle,
    # may be useful for computing (x, y) in unrolled strips.
    u = (1. - 2. * sinf * cosf) * cost
    v = -(theta * sint + 2. * sinf**2 * cost)
    w = sqrt(u**2 + v**2)
    u /= w
    v /= w
    return x, y, u, v


def get_cones(ncone=19):
    """return (lat, y, f, u, v) for ncone inscribed cones

    These divide globe into ncone cones spanning equal latitudes plus two
    polar disks with diameter equal to the common width of the latitude strips.
    That common width is dlat = 180/(ncone-1) degrees.  The first and last
    "cones" are always the polar disks.  The middle "cone" is a cylinder if
    ncone is odd (recommended).  The default ncone=19 produces 10 degree
    conical strips.

    Returns
    -------
    lat : ndarray shape(ncone+1)
        latitudes (radians) where cones are joined, which are latitude circles
        on the surface of the globe.
    y : ndarray shape(ncone+1)
        vertical coordinates at x=0 in unrolled flattened mapping corresponding
        to lat
    f : ndarray shape(ncone)
        fraction of full circle for unrolled latitude arc
    u : ndarray shape(ncone+1)
        coordinate normal to (x, y) plane corresponding to lat on surface of
        globe, that is, the 3D radius of the longitude circle
    v : ndarray shape(ncone+1)
        y coordinate of joint between cones on globe (y runs from south to
        north pole)
    """
    latmax = 0.5*(pi + pi/(ncone - 1))
    lat = linspace(-latmax, latmax, ncone + 1)
    v = sin(lat)
    lat[0], lat[-1] = -0.5*pi, 0.5*pi
    u = cos(lat)
    y = 0.*lat
    y[1:] = sqrt((u[1:] - u[:-1])**2 + (v[1:] - v[:-1])**2).cumsum()
    y -= y.mean()
    f = (u[:-1] - u[1:]) / (y[1:] - y[:-1])
    return lat, y, f, u, v


def cone_edges(ncone=19, nlon=360):
    """return edges for ncone conical strips at nlon+1 longitudes on (-180,180)

    Returns
    -------
    x : ndarray shape(ncone, 2, nlon+1)
        x coordinates of unrolled cone edges, x[:, 0] lower edge of cone i,
        x[: 1] upper edge of cone i
    y : ndarray shape(ncone, 2, nlon+1)
        y coordinates of unrolled cone edges
    """
    lat, y0, f, u, v = get_cones(ncone)
    phi = linspace(-pi, pi, nlon+1)
    x = zeros((ncone, 2, nlon+1))
    hfphi = 0.5 * f[:, None] * phi  # shape(ncone, nlon+1)
    hfphi = hfphi[:, None]  # shape(ncone, 1, nlon+1)
    shfphi = sin(hfphi)
    sinc_hfphi = sinc2(hfphi, shfphi)
    phiu = phi * u[:, None]  # shape(ncone+1, nlon+1)
    x[:, 0] = phiu[:-1]
    x[:, 1] = phiu[1:]
    x *= sinc_hfphi
    y = x + 0.  # copy x
    x *= cos(hfphi)
    y *= shfphi
    y[:, 0] += y0[:-1, None]
    y[:, 1] += y0[1:, None]
    return x, y


def cone_map(ncone, lon, lat=None):
    """return (x, y, icone) corresponding to (lon, lat) in radians"""
    from_map = lat is None
    if from_map:
        # Assume lon is an image with leading shape (nlat, nlon) in an
        # equirectangular projection.
        # Upper left corner is at (lon, lat) = (-180, +90) degrees.
        # Note that in image raster order, latitude is decreasing.
        nlat, nlon = lon.shape[:2]
        lon = linspace(-pi, pi, nlon+1)
        lat = linspace(0.5*pi, -0.5*pi, nlat+1)[:, None]
        # get conformable lon, lat of pixel centers in image
        lon = 0.5 * (lon[:-1] + lon[1:])
        lat = 0.5 * (lat[:-1] + lat[1:])
    else:
        lon, lat = asarray(lon, float), asarray(lat, float)  # assume conformal
        lon = (lon + pi) % (2*pi) - pi  # force -pi <= lon < pi
    lat0, y0, f, u, v = get_cones(ncone)
    bins = lat0 + 0.
    bins[0], bins[-1] = -1.e99, 1.e99  # extend latitude bins to infinity
    icone = digitize(lat, bins) - 1  # assume -pi/2 <= lat <= pi/2
    latc = 0.5 * (lat0[1:] + lat0[:-1])  # central latitude of each cone
    latc[0], latc[-1] = -0.5 * pi, 0.5 * pi  # polar disks special case
    uc = 0.5 * (u[1:] + u[:-1])
    uc[0] = uc[-1] = 0.  # polar disks are only half width
    yc = 0.5 * (y0[1:] + y0[:-1])
    yc[0], yc[-1] = y0[0], y0[-1]
    f, latc, uc, yc = f[icone], latc[icone], uc[icone], yc[icone]
    dy = sin(lat - latc)
    u = uc - f * dy  # f = -du/dy, u = radius to globe axis
    hfphi = 0.5 * f * lon
    shfphi = sin(hfphi)
    sinc_hfphi = sinc2(hfphi, shfphi)
    x = lon * u * sinc_hfphi
    x, y = x * cos(hfphi), yc + dy + x * shfphi
    icone = icone + zeros(x.shape, int)  # ensure icone same shape as x, y
    return x, y, icone


def great_circle(lon0=-110, lat0=50, ncone=19, npts=361):
    lon0, lat0 = lon0 * pi/180., lat0 * pi/180.
    eps = 0.0001  # tiny angle, but not too small
    if lat0 < 0.:
        # Ensure that 0 <= lat0 by rotating lon0 180 degrees if not
        lat0 = -lat0
        lon0 += pi
    lat0 = min(lat0, 0.5*pi-eps)  # now 0 <= lat0 < 0.5*pi - eps
    lon0 = (lon0 + pi) % (2.*pi) - pi  # now -pi <= lon0 < pi
    # let a = angle around great circle starting from lon0
    #    point on great circle is (x', y', z') = (cos(a), sin(a), 0)
    # taking (x, y) = (1, 0) at lon0,
    #    (x, y, z) = (x'*c - z'*s, y', x'*s + z'*c) = (x'*c, y', x'*s)
    #    (x', y', z') = (x*c + z*s, y, z*c - x*s)
    #      where c = cos(lat0), s = sin(lat0)
    # (x, y, z) = (c*cos(a), sin(a), s*cos(a))
    #           = (cos(lat)*cos(lon), cos(lat)*sin(lon), sin(lat))
    # (x', y', z') = (c*cos(lat)*cos(lon) + s*sin(lat), cos(lat)*sin(lon), 0)
    # This great circle reaches maximum lat=lat0 at lon=0, but we want
    # it to reach maximum lat at lon=lon0.  Hence lon=0 -> lon0, which means
    # that lon = -pi-lon0 -> -pi, our branch cut in the (x, y) plane.  What is
    # the value of a1 at lon1 = -pi - lon0?
    #    c1, s1 = cos(lon1), sin(lon1) = -cos(lon0), sin(lon0)
    #        z' = c*sin(lat) - s*cos(lat)*c1 = 0
    #    x', y' = c*cos(lat)*c1 + s*sin(lat), cos(lat)*s1
    #           = cos(a1),                    sin(a1)
    # Since cos(lat) >= 0, the equation for z'=0 determines lat1 at lon1:
    #    c*sin(l1) = s*cos(lat)*c1
    #    cos(lat), sin(lat) = (c, s*c1)/sqrt(c**2 + s**2*c1**2) = (c, s*c1)/p
    #    p = sqrt(c**2 + s**2*c1**2) = sqrt(1 - s**2*s1**2)
    #      = sqrt(1 - s**2 * sin(lon0)**2)
    #    (cos(a1), sin(a1)) = (c**2*c1 + s**2*c1, c*s1) / p = (c1/p, c*s1/p)
    #    a1 = atan2(c*s1, c1) = atan2(c*sin(lon0), -cos(lon0))
    c, s = cos(lat0), sin(lat0)  # Note that c, s both >= 0 since lat0 >= 0
    a = linspace(0., 2*pi, npts)
    a[0] += eps  # avoid hitting branch cut exactly
    a[-1] -= eps
    a += atan2(c*sin(lon0), -cos(lon0))
    z = cos(a)  # temporary
    lon = lon0 + arctan2(sin(a), c*z)
    z *= s
    lon = (lon + pi) % (2.*pi) - pi
    lat = arctan2(z, sqrt(1. - z*z))
    # lon now monotonically increases from -pi to pi
    if (lon[1:] - lon[:-1]).min() < 0.:
        raise RuntimeError("impossible great_circle problem - bug?")
    x, y, icone = cone_map(ncone, lon, lat)
    # Now must split (x, y) into chunks of constant icone, adding endpoints
    # at the breaks.  The break at the lon = -pi or pi branch cut has already
    # been handled, so all these interruptions are latitude bounds crossings.
    i0 = where(icone[1:] != icone[:-1])[0]  # cone boundary crossings where
    if len(i0) < 1:
        return [[x, y, icone[0]]]
    igap = array([i0, i0+1])  # gap is between i0, i0+1
    latmax = 0.5*(pi + pi/(ncone - 1))
    latbins = linspace(-latmax, latmax, ncone + 1)[1:-1]
    ilat = icone[igap].min(axis=0)
    latx = latbins[ilat]  # latitude of boundary being crossed by each gap
    gaplon, gaplat = lon[igap], lat[igap]
    gaplat0, gaplon0 = gaplat[0], gaplon[0]
    dlatx = gaplat[1] - gaplat0
    dphdth = (gaplon[1] - gaplon0) / dlatx
    lonx = gaplon0 + dphdth*(latx - gaplat0)  # approximate lon at crossing
    # get two points barely on opposite sides of each gap
    dlatx = array([-eps, eps])[:, None]
    dlatx = dlatx * where(dphdth < 0, -1., 1.)
    latx = latx + dlatx
    lonx = lonx + dphdth * dlatx
    xx, yy, iicone = cone_map(ncone, lonx, latx)
    # Split (lon, lat) into parts each spanning a single cone
    pcone = concatenate([icone[i0], icone[-1:]])
    parts, iprev, gnmax = [], 0, len(i0) - 1
    for g, i in enumerate(concatenate([i0+1, [npts]])):
        gp = max(g - 1, 0)
        gn = min(g + 1, gnmax)
        xp = concatenate([xx[1, gp:g], x[iprev:i], xx[0, g:gn]])
        yp = concatenate([yy[1, gp:g], y[iprev:i], yy[0, g:gn]])
        parts.append([xp, yp, pcone[g]])
        iprev = i
    return parts


def reproject(original, ncone=19, xygrid=None):
    x, y, icone = cone_map(ncone, original)
    nlat, nlon = original.shape[:2]
    if xygrid is None:
        # Default is 1400x1080 pixel image with height 5.0 (ncone=19).
        # - Chosen assuming original is 2160x1080, missing pixels in output
        #   will appear if xygrid is too fine.
        # Note that 1080p is output height of 3.6 inch at 300 dpi.
        #           1400 width is 4+2/3 inch at 300 dpi
        hwidth = 1400./1080. * 2.50  # 2.50 is half height
        xgrid = linspace(-hwidth, hwidth, 1401)
        ygrid = linspace(2.50, -2.50, 1081)
    else:
        xgrid, ygrid = xygrid
    nx, ny = len(xgrid) - 1, len(ygrid) - 1
    ix, iy = digitize(x, xgrid), digitize(y, ygrid)  # (ix, iy) is output pixel
    mask = (ix < 1) | (ix > nx) | (iy < 1) | (iy > ny)
    if mask.any():
        raise ValueError("{} points outside xygrid".format(mask.sum()))
    ix, iy = ix - 1, iy - 1
    ipix = ix + nx * iy  # output pixel index for each pixel in original
    pxbins = arange(nx * ny + 1) - 0.5
    # need input pixel lat and lon indices corresponding to ipix
    npix = histogram(ipix, pxbins)[0]  # count input pixels in each output pixel
    missing = npix == 0
    weight = 1. / (missing + npix)
    wlon = arange(nlon)
    wlat = arange(nlat)[:, None] + 0*wlon
    wlon = wlon + 0*wlat
    jlon = histogram(ipix, pxbins, weights=wlon)[0]
    jlat = histogram(ipix, pxbins, weights=wlat)[0]
    jlon = (jlon * weight + 0.499999).astype(int).reshape(ny, nx)
    jlat = (jlat * weight + 0.499999).astype(int).reshape(ny, nx)
    icone = icone[jlat, jlon]
    icone[missing.reshape(ny, nx)] = -1
    extent = -hwidth, hwidth, ygrid[-1], ygrid[0]
    return jlon, jlat, icone, extent


def get_parabolas(nparab=13, npts=181, rty=(240., 680., 300.)):
    # Idea is to make parabolas fit into same 1400 x 1080 space as polyconic.
    # Here, may as well use pixels as coordinates, with (0, 0) at the center,
    # and the plot limits intended to be (-700, 700) in x (-540, 540) in y.
    # Time will run from -680 to 680 to leave same 20 pixel margin as conic.
    # Ground extends from -240 to 240 and top parabola curves up by 300,
    # since quarter period of sine wave will have parabola height (pi/2)**2/2
    # or about 1.25, and 1.25*240 = 300.  This puts the ground at 240+300 = 540
    # at time 680, so the sky will clip on the top and bottom parabolas.
    rsurf, tmax, dymax = rty
    hdr = rsurf / (nparab - 1)
    r0 = linspace(-rsurf - hdr, rsurf + hdr, nparab + 1)
    rc = 0.5 * (r0[1:] + r0[:-1])  # center points of strips
    k0 = dymax / (rsurf * tmax**2)  # so h*(1 + k0*t**2) is parabola at r = h
    t = linspace(-tmax, tmax, npts)
    yc = rc[:, None] * (1. + k0 * t**2)  # central parabolas
    y = array([-hdr, hdr])[:, None, None] + yc  # shape(2, nparab, npts)
    # half of first and last parabolas are sky, not ground
    ysky = y[:, [0, -1]] + 0.  # shape(2, 2, npts)
    ysky[0, 1] += hdr
    ysky[1, 0] -= hdr
    # adjust first and last parabolas to only extend to surface
    y[1, -1] -= hdr
    y[0, 0] += hdr
    return t, y, ysky  # shapes: (npts,), (2, nparab, npts), (2, 2, npts)


def split_sine(t0=0., nparab=13, npts=181, rty=(240., 680., 300.)):
    rsurf, tmax, dymax = rty
    hdr = rsurf / (nparab - 1)
    r0 = linspace(-rsurf - hdr, rsurf + hdr, nparab + 1)
    rc = 0.5 * (r0[1:] + r0[:-1])  # center points of strips
    k0 = dymax / (rsurf * tmax**2)  # so h*(1 + k0*t**2) is parabola at r = h
    t = linspace(-tmax, tmax, npts)
    dyc = rc[:, None] * k0 * t**2  # central parabolas
    # Surface acceleration = rsurf * omega**2 = rsurf * 2*k0, so:
    omega = sqrt(2. * k0)
    wave = rsurf * cos(omega * (t - t0))  # ball dropped at t = t0
    # check: reaches center at t = pi/2/omega = tmax*sqrt(pi**2/8*rsurf/dymax)
    # and dymax/rsurf has been chosen near pi**2/8, so reaches center at tmax
    rbins = r0[1:-1]
    ip = digitize(wave, rbins)  # index of parabolic strip
    y = wave + dyc[ip, arange(npts)]
    # get gap crossings
    ix = where(ip[1:] != ip[:-1])[0]
    ipx0, ipx1 = ip[ix], ip[ix+1]  # strip to left, right of gap
    ipg = minimum(ipx0, ipx1)
    r0x = rbins[ipg]  # r(t=0) at crossing, shape(ngap,)
    dtdy = (t[ix+1] - t[ix]) / (wave[ix+1] - wave[ix])
    sgn = where(dtdy < 0., -1., 1.)
    eps = 0.01
    dy = array([-sgn*eps, sgn*eps])  # shape(2, ngap)
    tx = t[ix] + dtdy * (r0x - wave[ix] + dy)
    yx = rsurf * cos(omega * (tx - t0))
    yx += rc[[ipx0, ipx1]] * k0 * tx**2
    pparab = concatenate([ipx0, ip[-1:]])
    parts, iprev, gnmax = [], 0, len(ix)
    tprev = yprev = [[], []]
    for g, i in enumerate(concatenate([ix+1, [npts]])):
        gp = max(g - 1, 0)
        gn = min(g + 1, gnmax)
        tp = concatenate([tx[1, gp:g], t[iprev:i], tx[0, g:gn]])
        yp = concatenate([yx[1, gp:g], y[iprev:i], yx[0, g:gn]])
        parts.append([tp, yp, pparab[g]])
        iprev = i
    return parts, rsurf*cos(omega*t0)


In [7]:
# Plotting imports and utility functions

from PIL import Image
from matplotlib import rc
from matplotlib.pyplot import (clf, axes, axis, plot, text, arrow, savefig,
                               figure, fill_between, fignum_exists)
from matplotlib.colors import to_rgb
from matplotlib.animation import writers as mwriters

rc('lines', linewidth=1)
rc("figure", facecolor="white")
rc("axes", facecolor="white")

# World map image is derived from Natural Earth 1 image at:
# https://www.naturalearthdata.com/downloads/50m-natural-earth-1/
#   NE1_250M.png is NE1_50M_SR_W[2::5, 2::5] to reduce to 2160x1080
#   NE1_250M_dark.png has saturation increased, lightness decreased with Gimp
#     upper left is (lat=-180, lon=90)
# source data at https://github.com/nvkelso/natural-earth-vector
#world = array(Image.open("NE1_250M_dark.png"))  # shape(2160, 1080, 3) u1 RGB
# Atlas 1b.jpg adapted from
# https://www.paul-reed.co.uk/images/atlas1.jpg
world = array(Image.open("atlas1b.jpg"))  # shape(2160, 1080, 3) u1 RGB
poly_world = reproject(world)


In [ ]:
# Bouncing ball and deepest hole curved spacetime diagrams

def make_deep():
    global fig_eph, ax_eph
    if fig_deep is None:
        fig_deep = figure(2)
        fig_deep.set_figwidth(14./3.)
        fig_deep.set_figheight(3.6)
        fig_deep.set_dpi(300.)
        ax_deep = fig_deep.add_axes((0., 0., 1., 1.))
    return fig_deep, ax_deep


fig_deep = fig_newton = fig_bounce = None
#ground = "#e7ad84"  # from mountains in NE1_250M_dark.png
#sky = "#4badef"  # from continental shelf in NE1_250M_dark.png
#ground = "#b1927f"  # from mountains in atlas1b.png
#sky = "#3a5f79"  # from ocean in atlas1b.png
# Better to just go with simple named colors:
ground = "peru"
sky = "skyblue"


def deepest_hole(t, y, ysky, save=None, t0=None, bg="lightgray"):
    fig_deep, ax_deep = make_deep()
    ax_deep.clear()
    fig_deep.set_facecolor(bg)
    ax_deep.set_aspect("equal", adjustable="box")
    ax_deep.axis("off")
    for y12 in y.transpose([1, 0, 2]):
        ax_deep.fill_between(t, y12[0], y12[1], color=ground)
    for y12 in ysky.transpose([1, 0, 2]):
        ax_deep.fill_between(t, y12[0], y12[1], color=sky)
    ax_deep.set_xlim(-700, 700)
    ax_deep.set_ylim(-540, 540)
    if t0 is not None:
        parts, y0 = split_sine(t0)
        for p in parts:
            ax_deep.plot(*p[:2], c="k", lw=0.5)
        ax_deep.plot(0., y0, "o", c="k", ms=4)
    if save:
        if not isinstance(save, str):
            save = "fig_deepest"
        if t0 is not None and save == 1:
            save += "{}".format(int(t0))
        save += ".png"
        savefig(save, dpi=300, facecolor=bg)


def deepest_movie(with_dot=True, name="fig_deepdrop.mp4"):
    rsurf, tmax, dymax = 240., 680., 300.
    k0 = dymax / (rsurf * tmax**2)  # so h*(1 + k0*t**2) is parabola at r = h
    omega = sqrt(2. * k0)
    period = 2.*pi / omega
    ffmpeg = mwriters["ffmpeg"]
    writer = ffmpeg(fps=48, metadata=dict(title="  ",
        comment="ball dropping through center of earth"))
    dpi = 300
    # Write 241 frame 5 second movie of a single rotation.
    parabs = get_parabolas()
    fig_deep, ax_deep = make_deep()
    with writer.saving(fig_deep, name, dpi):
        t0, dt0 = 0., period / 240.
        for i in range(241):
            deepest_hole(*parabs, 0, -t0)
            writer.grab_frame()
            t0 += dt0


def make_newton():
    global fig_newton, ax_newton
    if fig_newton is None:
        fig_newton = figure(3)
        fig_newton.set_figwidth(14./3.)
        fig_newton.set_figheight(2.0)
        fig_newton.set_dpi(300.)
        ax_newton = fig_newton.add_axes((0., 0., 1., 1.))
    return fig_newton, ax_newton


def newton_hole(save=None, t0=None, rty=(240., 680., 300.),
                nparab=13, npts=181, bg="lightgray"):
    fig_newton, ax_newton = make_newton()
    rsurf, tmax, dymax = rty
    hdr = rsurf / (nparab - 1)
    ysky = array([[-rsurf - hdr, -rsurf], [rsurf, rsurf + hdr]])
    yground = array([-rsurf, rsurf])
    k0 = dymax / (rsurf * tmax**2)  # so h*(1 + k0*t**2) is parabola at r = h
    omega = sqrt(2. * k0)
    t = array([-tmax, tmax])
    ax_newton.clear()
    fig_newton.set_facecolor(bg)
    ax_newton.set_aspect("equal", adjustable="box")
    ax_newton.axis("off")
    ax_newton.fill_between(t, [-rsurf]*2, [rsurf]*2, color=ground)
    for y0, y1 in [[-rsurf - hdr, -rsurf], [rsurf, rsurf + hdr]]:
        ax_newton.fill_between(t, [y0]*2, [y1]*2, color=sky)
    t = linspace(-tmax, tmax, npts)
    t00 = 0. if t0 is None else t0
    wave = rsurf * cos(omega * (t - t00))
    if t0 is not None:
        ax_newton.plot(t, wave, c="k", lw=0.5)
        ax_newton.plot(0., rsurf * cos(omega * t0), "o", c="k", ms=4)
    ax_newton.set_xlim(-700, 700)
    ax_newton.set_ylim(-300, 300)
    if save:
        if not isinstance(save, str):
            save = "fig_newton"
        if t0 is not None and save == 1:
            save += "{}".format(int(t0))
        save += ".png"
        savefig(save, dpi=300, facecolor=bg)


def make_bounce():
    global fig_bounce, ax_bounce
    if fig_bounce is None:
        fig_bounce = figure(4)
        fig_bounce.set_dpi(300.)
        ax_bounce = fig_bounce.add_axes((0., 0., 1., 1.))
    return fig_bounce, ax_bounce


def draw_bounces(einstein=False, save=None,
                 h1=0.5, nhb=4.5, npts=40, bg="lightgray"):
    fig_bounce, ax_bounce = make_bounce()
    # Take g=2, so that y = t**2.  A fall from h takes sqrt(h), a half bounce.
    imax = int(npts * nhb + 0.5)  # max time in units of thb / npts
    balls = []
    for h in (1., h1):
        thb = sqrt(h)  # time for one half bounce
        nhbh = nhb / thb  # number of half bounces for this h
        imax = int(npts * nhbh + 0.5)  # max time in units of thb / npts
        t = arange(imax+1) * thb / npts  # time in units of h=1 half bounces
        # peaks at 0, t[2*npts], t[4*npts], ...
        # bounces at t[npts], t[3*npts], t[5*npts], etc.
        tpeaks = t[::2*npts]
        tbounces = tpeaks + thb
        y = h - h * ((t - tpeaks[digitize(t, tbounces)]) / thb)**2
        if einstein:
            y += (t - 0.5*nhb)**2
        balls.append([t, y])
    ax_bounce.clear()
    fig_bounce.set_facecolor(bg)
    ax_bounce.axis("off")
    yextra = 0.1  # thickness of ground below y=0 and sky above y=1
    tmax = max([b[0][-1] for b in balls])
    if not einstein:
        t, y1 = array([0, tmax]), zeros(2)
        ymax = 1. + (tmax - 0.5*nhb)**2
        # ax_bounce.set_ylim(-0.5*ymax - hmargin, 0.5*ymax + hmargin)
    else:
        t = linspace(0, tmax, int(nhb*npts) + 1)
        y1 = (t - 0.5*nhb)**2
    ymin, ymax = -yextra, 1. + y1.max() + yextra
    ymaxe = 1. + (tmax - 0.5*nhb)**2 + yextra
    margin = 0.01
    dtm = margin * tmax
    dyme = margin * (ymaxe - ymin)
    aspect = 0.75 * (ymax - ymin + 2.*dyme) / (ymaxe - ymin + 2.*dyme)
    fig_bounce.set_figwidth(6.4)
    fig_bounce.set_figheight(6.4 * aspect)
    ax_bounce.set_xlim(-dtm, tmax + dtm)
    ax_bounce.set_ylim(ymin - dyme, ymax + dyme)
    ax_bounce.fill_between(t, y1-yextra, y1, color=ground)
    ax_bounce.fill_between(t, y1, y1+1.+yextra, color=sky)
    for b, c in zip(balls, ("black", "yellow")):
        ax_bounce.plot(*b, lw=1, c=c)
    if save:
        if not isinstance(save, str):
            save = "fig_{}bounce".format("e" if einstein else "n")
        save += ".png"
        savefig(save, dpi=300, facecolor=bg)


draw_bounces(0, 0)
# draw_bounces(1, 0)
# deepest_hole(*get_parabolas(), 0, -400)
# newton_hole(0, -400)
# deepest_movie()


In [ ]:
# Interrupted polyconic world map figures

fig_cones = figure(1)
fig_cones.set_figwidth(14./3.)
fig_cones.set_figheight(3.6)
fig_cones.set_dpi(300.)
ax_cones = fig_cones.add_axes((0., 0., 1., 1.))

# "white" background OK for NE1_250M_dark.png
# "lightgray" works better for atlas1b.jpg
bg_color = "lightgray"


def plot_flattened(original, jlon, jlat, icone, extent, dj=0, bg="white",
                   save=None, with_gc=True):
    if dj:
        # Note that actual width of original is 2160 pixels for -pi to pi
        # so -1080 < dj < 1080 is actual range mapping to -180 < lon < 180
        # or dj = 6 corresponds to a 1 degree rotation
        # dj = -450 lines up the Americas pretty well
        nx = original.shape[1]
        jlon = (jlon + dj) % nx
        if jlon.min() < 0:
            jlon = (jlon + nx) % nx
    flat = original[jlat, jlon]
    flat[icone < 0] = tuple(int(x * 255 + 0.5) for x in to_rgb(bg))
    ax_cones.clear()
    fig_cones.set_facecolor(bg)
    ax_cones.set_aspect("equal", adjustable="box")
    ax_cones.axis("off")
    ax_cones.imshow(flat, extent=extent)
    if with_gc:
        parts = great_circle(-110 - dj/6)
        for p in parts:
            plot(*p[:2], c='k', lw=0.5)
    if save:
        if not isinstance(save, str):
            save = "fig_polycone"
        if dj:
            nx = original.shape[1]
            dj = dj % nx
            if dj < 0:
                dj += nx
            save += "{}".format(dj)
        save += ".png"
        savefig(save, dpi=300, facecolor=bg)


def spinning_movie(with_gc=True, name="fig_polyspin.mp4"):
    ffmpeg = mwriters["ffmpeg"]
    writer = ffmpeg(fps=48, metadata=dict(title="  ",
        comment="polyconic projection of spinning globe"))
    dpi = 300
    # Write 241 frame 5 second movie of a single rotation.
    with writer.saving(fig_cones, name, dpi):
        for i in range(241):
            dj = 2160-9*i
            plot_flattened(world, *poly_world, dj=2160-9*i, bg=bg_color)
            if with_gc:
                parts = great_circle(-110 - dj/6)
                for p in parts:
                    plot(*p[:2], c='k', lw=0.5)
            writer.grab_frame()


plot_flattened(world, *poly_world, dj=0, save=0, bg=bg_color)
# spinning_movie()


In [ ]:
from ephemeris import ephemeris, jd2t

ephemeris.set_model(planets=("mercury", "venus", "earth", "mars"))

# Use time of first gravitational wave detected by LIGO as time origin.
t0 = jd2t("2015-09-14 09:50:45", 0)

xyzp, tp = ephemeris.one_orbit(t0)

fig_eph = None


def make_eph(bg="lightgray"):
    global fig_eph, ax_eph
    if fig_eph is None:
        fig_eph = figure(5)
        # fig_eph.set_figwidth(14./3.)
        # fig_eph.set_figheight(3.6)
        fig_eph.set_dpi(300.)
        ax_eph = fig_eph.add_axes((0., 0., 1., 1.))
    fig_eph.set_facecolor(bg)
    return fig_eph, ax_eph


def get_orbits(ntps=1025):
    tq = linspace(0., 2.*(tp.max() - t0), 1025)
    xyzq = xyzp[..., :1] + 0. * tq
    for i in range(xyzq.shape[1]):
        t = tp[i] - t0
        xyzq[:, i] = [interp(tq % t.max(), t, w) for w in xyzp[:, i]]
    return xyzq, tq
        


def quick_eph(earth=False, bg="gray"):
    fig_eph, ax_eph = make_eph(bg)
    ax_eph.clear()
    ax_eph.axis("off")
    ax_eph.set_aspect("equal", adjustable="box")
    colors = ["yellow", "red", "blue", "white", "tan"]
    bg = array(to_rgb(bg))
    xc, yc = xyzq[:2, 2] if earth else zeros((2, 1))
    for x, y, _ in xyzq.transpose([1, 0, 2]):
        c = colors.pop()
        clo = tuple((array(to_rgb(c)) + 2.*bg) / 3.)
        ax_eph.plot(x - xc, y - yc, c=clo)
        ax_eph.plot(x[-1] - xc[-1], y[-1] - yc[-1], "o", c=c, ms=4)
    c = colors.pop()
    clo = tuple((array(to_rgb(c)) + 2.*bg) / 3.)
    ax_eph.plot(-xc, -yc, c=clo)
    ax_eph.plot(-xc[-1], -yc[-1], "o", ms=4, c=c)
    ax_eph.set_xlim(-2.72, 2.12)
    ax_eph.set_ylim(-1.93, 2.785)


xyzq, tq = get_orbits()
quick_eph(1)
